In [1]:
# Configuração
from pathlib import Path
import os
import sys
import json
import time
import random

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/AMMCI_PRF")
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
METRICS_DIR = OUTPUT_DIR / "metrics"
MODEL_DIR = OUTPUT_DIR / "models"

METRICS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Projeto:", PROJECT_ROOT)
print("Dados:", DATA_DIR)

Projeto: D:\Workspace\ws-uem\acidentes-rodoviarios-ml
Dados: D:\Workspace\ws-uem\acidentes-rodoviarios-ml\data


# 05 — Experimentos temporais

## Objetivo

Comparar quatro estratégias temporais:

- **M0:** treinamento do zero em D0;
- **MFT:** começa nos pesos de M0 e continua em D1;
- **MRT:** treinamento do zero em D0 + D1;
- **MREC:** treinamento do zero somente em D1.

A MLP vem do notebook 04 e cada estratégia usa três seeds. D2 permanece reservado para o final.

Os resultados são salvos em `outputs/`.

In [2]:
import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    average_precision_score,
)

import tensorflow as tf
from tensorflow import keras

SEEDS = [42, 123, 2026]
TARGET = "classificacao_acidente"

CATEGORICAL_FEATURES = [
    "dia_semana", "uf", "br", "causa_acidente", "tipo_acidente",
    "fase_dia", "sentido_via", "condicao_metereologica",
    "tipo_pista", "tracado_via", "uso_solo",
]

NUMERIC_FEATURES = [
    "hora", "mes", "km", "latitude", "longitude", "veiculos",
]

FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

CLASS_NAMES = [
    "Sem Vítimas",
    "Com Vítimas Feridas",
    "Com Vítimas Fatais",
]

CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

CONFIG_PATH = MODEL_DIR / "mlp_config.json"
if not CONFIG_PATH.exists():
    raise FileNotFoundError("Execute primeiro o notebook 04.")

selection_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
base_config = selection_config["architecture"]
TRACADO_ENCODING = selection_config["preprocessing"]["tracado_encoding"]
FINAL_EPOCHS = int(base_config["final_epochs"])

print("MLP selecionada:", selection_config["selected_config_name"])
print("Épocas finais:", FINAL_EPOCHS)
print("Seeds:", SEEDS)

MLP selecionada: Multi-hot — combinada
Épocas finais: 10
Seeds: [42, 123, 2026]


In [3]:
# Dados e funções auxiliares
def load_split(name):
    path = DATA_DIR / f"{name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Não encontrei {path}. Execute primeiro o notebook 01.")

    df = pd.read_csv(
        path,
        sep=";",
        encoding="utf-8",
        dtype={"br": "string"},
        parse_dates=["timestamp"],
    )

    return df.sort_values("timestamp", kind="stable").reset_index(drop=True)


def build_preprocessor():
    categories = [
        c for c in CATEGORICAL_FEATURES
        if c != "tracado_via" or TRACADO_ENCODING == "categorical"
    ]

    transformers = [
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
            ]),
            categories,
        ),
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            NUMERIC_FEATURES,
        ),
    ]

    if TRACADO_ENCODING == "multihot":
        transformers.append((
            "tracado",
            CountVectorizer(
                token_pattern=r"(?u)[^;]+",
                lowercase=False,
                binary=True,
                dtype=np.float32,
            ),
            "tracado_via",
        ))

    return ColumnTransformer(transformers, sparse_threshold=1.0)


def dense(X):
    X = X.astype(np.float32)
    return X.toarray() if sparse.issparse(X) else np.asarray(X, dtype=np.float32)


def encode_y(df):
    return df[TARGET].map(CLASS_TO_ID).to_numpy(dtype=np.int64)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def class_weights_for(y):
    alpha = float(base_config["class_weight_alpha"])

    if alpha == 0:
        return None

    counts = np.bincount(y, minlength=len(CLASS_NAMES))
    weights = (len(y) / (len(CLASS_NAMES) * counts)) ** alpha
    weights /= np.average(weights, weights=counts)

    return {i: float(w) for i, w in enumerate(weights)}


def build_mlp(input_dim, seed, learning_rate=None):
    set_seed(seed)

    lr = base_config["learning_rate"] if learning_rate is None else learning_rate

    model = keras.Sequential([
        keras.layers.Input(shape=(input_dim,))
    ])

    for units in base_config["hidden_units"]:
        model.add(
            keras.layers.Dense(
                units,
                activation="relu",
                kernel_regularizer=keras.regularizers.l2(base_config["l2"]),
            )
        )

        if base_config["dropout"] > 0:
            model.add(keras.layers.Dropout(base_config["dropout"]))

    model.add(keras.layers.Dense(len(CLASS_NAMES), activation="softmax"))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model


def fit_model(model, X, y, epochs):
    start = time.perf_counter()

    model.fit(
        X,
        y,
        epochs=epochs,
        batch_size=base_config["batch_size"],
        class_weight=class_weights_for(y),
        shuffle=True,
        verbose=0,
    )

    return time.perf_counter() - start


def evaluate(model, X, y, name, seed, evaluation_set, train_seconds):
    proba = model.predict(X, batch_size=4096, verbose=0)
    pred = proba.argmax(axis=1)

    row = {
        "modelo": name,
        "seed": seed,
        "evaluation_set": evaluation_set,
        "accuracy": accuracy_score(y, pred),
        "macro_precision": precision_score(
            y, pred, labels=range(3), average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            y, pred, labels=range(3), average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            y, pred, labels=range(3), average="macro", zero_division=0
        ),
        "mcc": matthews_corrcoef(y, pred),
        "macro_pr_auc": average_precision_score(
            np.eye(3)[y], proba, average="macro"
        ),
        "tempo_treino_s": train_seconds,
        "n_parametros": model.count_params(),
    }

    return row, pred, proba


class MacroF1Stopping(keras.callbacks.Callback):
    def __init__(self, X_val, y_val, patience=5):
        super().__init__()
        self.X_val = X_val
        self.y_val = y_val
        self.patience = patience
        self.best = -np.inf
        self.best_epoch = 0
        self.wait = 0
        self.best_weights = None

    def on_epoch_end(self, epoch, logs=None):
        pred = self.model.predict(
            self.X_val, batch_size=4096, verbose=0
        ).argmax(axis=1)

        score = f1_score(
            self.y_val,
            pred,
            labels=range(3),
            average="macro",
            zero_division=0,
        )

        if logs is not None:
            logs["val_macro_f1"] = score

        if score > self.best + 1e-4:
            self.best = score
            self.best_epoch = epoch + 1
            self.best_weights = self.model.get_weights()
            self.wait = 0
        else:
            self.wait += 1

            if self.wait >= self.patience:
                self.model.stop_training = True

    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)


d0 = load_split("d0")
d1 = load_split("d1")

print("D0:", len(d0), "|", d0.timestamp.min(), "→", d0.timestamp.max())
print("D1:", len(d1), "|", d1.timestamp.min(), "→", d1.timestamp.max())

D0: 87838 | 2023-01-01 00:01:00 → 2024-04-17 07:50:00
D1: 29279 | 2024-04-17 07:50:00 → 2024-09-08 15:30:00


## 1. M0 — histórico

M0 é treinado em D0 e avaliado em D1 antes de qualquer atualização.

In [4]:
m0_preprocessor = build_preprocessor()

X0 = dense(m0_preprocessor.fit_transform(d0[FEATURES]))
X1 = dense(m0_preprocessor.transform(d1[FEATURES]))

y0 = encode_y(d0)
y1 = encode_y(d1)

m0_models = {}
m0_times = {}
degradation_rows = []
training_times = []

for seed in SEEDS:
    model = build_mlp(X0.shape[1], seed)
    seconds = fit_model(model, X0, y0, FINAL_EPOCHS)

    m0_models[seed] = model
    m0_times[seed] = seconds

    training_times.append({
        "modelo": "M0",
        "seed": seed,
        "phase": "final",
        "n_train": len(y0),
        "epochs": FINAL_EPOCHS,
        "tempo_etapa_s": seconds,
        "n_parametros": model.count_params(),
    })

    if seed == 42:
        model.save_weights(MODEL_DIR / "m0_seed_42.weights.h5")

    row, _, _ = evaluate(
        model, X1, y1, "M0", seed, "D1", seconds
    )
    degradation_rows.append(row)

display(pd.DataFrame(degradation_rows).round(4))

,modelo,seed,evaluation_set,accuracy,macro_precision,macro_recall,macro_f1,mcc,macro_pr_auc,tempo_treino_s,n_parametros
0,M0,42,D1,0.7544,0.5803,0.5727,0.5758,0.3512,0.5969,12.2435,18499
1,M0,123,D1,0.7555,0.5792,0.5696,0.5703,0.3479,0.5947,12.9816,18499
2,M0,2026,D1,0.7574,0.5816,0.5658,0.5706,0.3468,0.5944,9.6227,18499


## 2. Escolha do fine-tuning em D1

Os primeiros 90% de D1 treinam; os últimos 10% selecionam o fine-tuning.

Depois, o MFT final usa todo D1.

In [5]:
# A intensidade do fine-tuning é escolhida apenas dentro de D1.
boundary = d1.iloc[int(len(d1) * 0.90)]["timestamp"]

d1_train = d1[d1.timestamp < boundary].copy()
d1_val = d1[d1.timestamp >= boundary].copy()

Xft = dense(m0_preprocessor.transform(d1_train[FEATURES]))
Xfv = dense(m0_preprocessor.transform(d1_val[FEATURES]))

yft = encode_y(d1_train)
yfv = encode_y(d1_val)

MFT_CONFIGS = [
    {
        "name": "MFT-A",
        "learning_rate": base_config["learning_rate"] / 10,
        "epochs": 20,
    },
    {
        "name": "MFT-B",
        "learning_rate": base_config["learning_rate"] / 20,
        "epochs": 40,
    },
]

mft_rows = []
mft_history_rows = []

for config in MFT_CONFIGS:
    for seed in SEEDS:
        model = build_mlp(
            X0.shape[1],
            seed,
            learning_rate=config["learning_rate"],
        )

        model.set_weights(m0_models[seed].get_weights())

        stopper = MacroF1Stopping(Xfv, yfv, patience=5)

        start = time.perf_counter()

        history = model.fit(
            Xft,
            yft,
            validation_data=(Xfv, yfv),
            validation_batch_size=4096,
            epochs=config["epochs"],
            batch_size=base_config["batch_size"],
            class_weight=class_weights_for(yft),
            shuffle=True,
            verbose=0,
            callbacks=[stopper],
        )

        seconds = time.perf_counter() - start

        row, _, _ = evaluate(
            model,
            Xfv,
            yfv,
            config["name"],
            seed,
            "D1_validation",
            seconds,
        )

        row["best_epoch"] = stopper.best_epoch
        row["learning_rate"] = config["learning_rate"]
        mft_rows.append(row)

        for epoch in range(len(history.history["loss"])):
            mft_history_rows.append({
                "modelo": config["name"],
                "seed": seed,
                "epoch": epoch + 1,
                "loss": history.history["loss"][epoch],
                "val_loss": history.history["val_loss"][epoch],
                "accuracy": history.history["accuracy"][epoch],
                "val_accuracy": history.history["val_accuracy"][epoch],
                "val_macro_f1": history.history["val_macro_f1"][epoch],
            })

mft_selection = pd.DataFrame(mft_rows)

mft_summary = (
    mft_selection
    .groupby("modelo")
    .macro_f1
    .agg(["mean", "std"])
    .reset_index()
    .rename(columns={
        "mean": "macro_f1_mean",
        "std": "macro_f1_std",
    })
    .sort_values(
        ["macro_f1_mean", "macro_f1_std", "modelo"],
        ascending=[False, True, True],
    )
)

SELECTED_MFT = mft_summary.iloc[0]["modelo"]

selected_mft_config = dict(
    next(c for c in MFT_CONFIGS if c["name"] == SELECTED_MFT)
)

selected_mft_config["final_epochs"] = max(
    1,
    int(np.rint(
        mft_selection.loc[
            mft_selection.modelo == SELECTED_MFT,
            "best_epoch",
        ].median()
    )),
)

print("Fine-tuning escolhido:", selected_mft_config)
display(mft_summary.round(4))

Fine-tuning escolhido: {'name': 'MFT-A', 'learning_rate': 0.0001, 'epochs': 20, 'final_epochs': 6}


,modelo,macro_f1_mean,macro_f1_std
0,MFT-A,0.5530,0.0038
1,MFT-B,0.5502,0.0037


## 3. MFT, MRT e MREC finais

In [6]:
# MFT — parte dos pesos de M0 e continua em todo D1
mft_models = {}
mft_times = {}

for seed in SEEDS:
    model = build_mlp(
        X0.shape[1],
        seed,
        learning_rate=selected_mft_config["learning_rate"],
    )

    model.set_weights(m0_models[seed].get_weights())

    seconds = fit_model(
        model,
        X1,
        y1,
        selected_mft_config["final_epochs"],
    )

    mft_models[seed] = model
    mft_times[seed] = seconds

    training_times.append({
        "modelo": "MFT",
        "seed": seed,
        "phase": "final",
        "n_train": len(y1),
        "epochs": selected_mft_config["final_epochs"],
        "tempo_etapa_s": seconds,
        "n_parametros": model.count_params(),
    })

    if seed == 42:
        model.save_weights(MODEL_DIR / "mft_seed_42.weights.h5")


# MRT — novo modelo treinado do zero em D0 + D1
# MREC — novo modelo treinado do zero somente em D1
d0d1 = (
    pd.concat([d0, d1], ignore_index=True)
    .sort_values("timestamp", kind="stable")
    .reset_index(drop=True)
)

other_models = {}
other_preprocessors = {}
other_times = {}

for name, frame in [("MRT", d0d1), ("MREC", d1)]:
    prep = build_preprocessor()
    X = dense(prep.fit_transform(frame[FEATURES]))
    y = encode_y(frame)

    other_preprocessors[name] = prep

    for seed in SEEDS:
        model = build_mlp(X.shape[1], seed)
        seconds = fit_model(model, X, y, FINAL_EPOCHS)

        other_models[(name, seed)] = model
        other_times[(name, seed)] = seconds

        training_times.append({
            "modelo": name,
            "seed": seed,
            "phase": "final",
            "n_train": len(y),
            "epochs": FINAL_EPOCHS,
            "tempo_etapa_s": seconds,
            "n_parametros": model.count_params(),
        })

print("Treinos concluídos. Agora D2 pode ser aberto.")

Treinos concluídos. Agora D2 pode ser aberto.


## 4. Avaliação final em D2

D2 é usado somente para a avaliação final.

In [7]:
# D2 é carregado somente após o congelamento de todas as estratégias.
d2 = load_split("d2")
y2 = encode_y(d2)

X2 = {
    "M0": dense(m0_preprocessor.transform(d2[FEATURES])),
}

X2["MFT"] = X2["M0"]

for name, prep in other_preprocessors.items():
    X2[name] = dense(prep.transform(d2[FEATURES]))

final_rows = []
prediction_frames = []

for seed in SEEDS:
    candidates = [
        ("M0", m0_models[seed], m0_times[seed]),
        ("MFT", mft_models[seed], m0_times[seed] + mft_times[seed]),
        ("MRT", other_models[("MRT", seed)], other_times[("MRT", seed)]),
        ("MREC", other_models[("MREC", seed)], other_times[("MREC", seed)]),
    ]

    for name, model, total_seconds in candidates:
        row, pred, proba = evaluate(
            model,
            X2[name],
            y2,
            name,
            seed,
            "D2",
            total_seconds,
        )

        final_rows.append(row)

        if name == "M0":
            degradation_rows.append(row.copy())

        prediction_frames.append(pd.DataFrame({
            "row_index": np.arange(len(d2)),
            "timestamp": d2.timestamp,
            "modelo": name,
            "seed": seed,
            "y_true": y2,
            "y_pred": pred,
            "proba_0": proba[:, 0],
            "proba_1": proba[:, 1],
            "proba_2": proba[:, 2],
        }))

final_runs = pd.DataFrame(final_rows)
predictions = pd.concat(prediction_frames, ignore_index=True)

display(final_runs.round(4))

,modelo,seed,evaluation_set,accuracy,macro_precision,macro_recall,macro_f1,mcc,macro_pr_auc,tempo_treino_s,n_parametros
0,M0,42,D2,0.7621,0.5699,0.5557,0.5616,0.3326,0.5792,12.2435,18499
1,MFT,42,D2,0.7707,0.5837,0.5587,0.5655,0.3420,0.5801,15.8665,18499
2,MRT,42,D2,0.7651,0.5764,0.5614,0.5650,0.3355,0.5815,10.9018,18627
3,MREC,42,D2,0.7715,0.5792,0.5272,0.5466,0.3183,0.5672,4.0560,17475
4,M0,123,D2,0.7644,0.5735,0.5608,0.5620,0.3363,0.5760,12.9816,18499
5,MFT,123,D2,0.7708,0.5850,0.5598,0.5660,0.3420,0.5789,16.7173,18499
6,MRT,123,D2,0.7713,0.5853,0.5575,0.5674,0.3421,0.5841,12.4254,18627
7,MREC,123,D2,0.7671,0.5778,0.5397,0.5472,0.3176,0.5639,3.8336,17475
8,M0,2026,D2,0.7674,0.5780,0.5546,0.5621,0.3339,0.5787,9.6227,18499
9,MFT,2026,D2,0.7688,0.5813,0.5577,0.5636,0.3368,0.5819,12.9335,18499


## 5. Persistência dos resultados

Métricas, previsões, pesos e tempos são salvos para o notebook 06.

In [8]:
METRICS = [
    "accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "mcc",
    "macro_pr_auc",
    "tempo_treino_s",
]

summary = (
    final_runs
    .groupby("modelo")[METRICS]
    .agg(["mean", "std"])
)

summary.columns = [
    f"{metric}_{stat}"
    for metric, stat in summary.columns
]

temporal_summary = (
    summary
    .reindex(["M0", "MFT", "MRT", "MREC"])
    .reset_index()
)

m0_f1 = temporal_summary.loc[
    temporal_summary.modelo == "M0",
    "macro_f1_mean",
].iloc[0]

temporal_summary["delta_macro_f1_vs_m0"] = (
    temporal_summary["macro_f1_mean"] - m0_f1
)

# Salvar somente os artefatos realmente usados depois.
final_runs.to_csv(METRICS_DIR / "temporal_runs.csv", index=False)
temporal_summary.to_csv(METRICS_DIR / "temporal_summary.csv", index=False)
pd.DataFrame(degradation_rows).to_csv(
    METRICS_DIR / "m0_degradation.csv", index=False
)

mft_selection.to_csv(
    METRICS_DIR / "mft_selection_runs.csv", index=False
)
mft_summary.to_csv(
    METRICS_DIR / "mft_selection_summary.csv", index=False
)
pd.DataFrame(mft_history_rows).to_csv(
    METRICS_DIR / "mft_selection_history.csv", index=False
)

pd.DataFrame(training_times).to_csv(
    METRICS_DIR / "training_times.csv", index=False
)

predictions.to_csv(
    METRICS_DIR / "d2_predictions.csv.gz",
    index=False,
    compression="gzip",
)

(MODEL_DIR / "selected_mft_config.json").write_text(
    json.dumps(selected_mft_config, indent=2),
    encoding="utf-8",
)

display(
    temporal_summary[
        [
            "modelo",
            "accuracy_mean",
            "macro_f1_mean",
            "macro_f1_std",
            "macro_recall_mean",
            "mcc_mean",
            "macro_pr_auc_mean",
            "delta_macro_f1_vs_m0",
        ]
    ].round(4)
)

print("Pronto. Execute o notebook 06.")

,modelo,accuracy_mean,macro_f1_mean,macro_f1_std,macro_recall_mean,mcc_mean,macro_pr_auc_mean,delta_macro_f1_vs_m0
0,M0,0.7646,0.5619,0.0003,0.5570,0.3343,0.5780,0.0000
1,MFT,0.7701,0.5650,0.0013,0.5587,0.3403,0.5803,0.0031
2,MRT,0.7680,0.5645,0.0031,0.5577,0.3370,0.5821,0.0026
3,MREC,0.7697,0.5471,0.0004,0.5352,0.3180,0.5664,-0.0148


Pronto. Execute o notebook 06.
